# Oscar winners for the AWARD_Director table
Written by: Capleton

This notebook makes a clean CSV file that fits the table AWARD_Director in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [23]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## STEP 2: read the file
The original file stays unchanged. If something goes wrong in the database, we can always build the table again from the full data.

In [24]:
df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")

print("Rows and columns:", df.shape)
print(df.head())

Rows and columns: (10889, 7)
   year_film  year_ceremony  ceremony category                 name              film  winner
0       1927           1928         1    ACTOR  Richard Barthelmess         The Noose   False
1       1927           1928         1    ACTOR        Emil Jannings  The Last Command    True
2       1927           1928         1  ACTRESS       Louise Dresser   A Ship Comes In   False
3       1927           1928         1  ACTRESS         Janet Gaynor        7th Heaven    True
4       1927           1928         1  ACTRESS       Gloria Swanson    Sadie Thompson   False


## STEP 3: keep only the winners with a film
The database should only hold data that answers our research questions. An award without a film can never be linked to a row in MOVIE, so it has no place in a table with a foreign key to MOVIE.

In [25]:
winners = df[df["winner"] == True]
winners = winners.dropna(subset=["film"])

print("Rows before:", len(df))
print("Rows after:", len(winners))

Rows before: 10889
Rows after: 2172


## STEP 4: keep only the directing awards
In our ERD every type of award has its own table. AWARD_Director has a foreign key to DIRECTOR, so it may only hold Oscars that belong to a director.

In [26]:
award_director = winners[winners["category"].str.startswith("DIRECTING")].copy()

print("Rows after:", len(award_director))
print("Categories:", list(award_director["category"].unique()))

Rows after: 97
Categories: ['DIRECTING (Comedy Picture)', 'DIRECTING (Dramatic Picture)', 'DIRECTING']


## STEP 5: rename the columns
The column names in the CSV must be the same as in the table in PostgreSQL. Otherwise pgAdmin does not know which value goes into which column.

In [27]:
award_director = award_director.rename(columns={"name": "name_nominee", "film": "film_title"})

print(list(award_director.columns))

['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']


## STEP 6: clean the film titles and make the match key
Some old titles in the source data contain a ;. It is a leftover and not part of the real title, so we remove it to keep the data in the database clean.

film_title_clean is the title in small letters, a hyphen becomes a space, dots are removed and double spaces become one. film_title stays as it is, so the real title with capitals is still stored.

match_key is film_title_clean plus the year of the film, for example no country for old men_2007. The database needs it to fill the foreign key movie_id: the Oscar data has no movie_id, so we look up this key in MOVIE. We add the year because different films can have the same title, like a remake. The year of the film is year_ceremony minus 1, because the Oscars are given the year after the film came out. MOVIE makes the same key from its title and the year of its release date.

In [28]:
award_director["film_title"] = award_director["film_title"].str.replace(";", "").str.strip()
award_director["film_title_clean"] = award_director["film_title"].str.lower().str.replace("-", " ", regex=False).str.replace(".", "", regex=False).str.split().str.join(" ")

award_director["match_key"] = award_director["film_title_clean"] + "_" + (award_director["year_ceremony"] - 1).astype(str)

print(award_director[["film_title", "film_title_clean", "match_key"]].head())
print("Titles with ; left:", award_director["film_title"].str.contains(";").sum())

                         film_title                film_title_clean                            match_key
11              Two Arabian Knights             two arabian knights             two arabian knights_1927
13                       7th Heaven                      7th heaven                      7th heaven_1927
59                  The Divine Lady                 the divine lady                 the divine lady_1928
98   All Quiet on the Western Front  all quiet on the western front  all quiet on the western front_1929
139                          Skippy                          skippy                          skippy_1930
Titles with ; left: 0


## STEP 7: sort the table
A fixed order gives every award the same director_award_id every time. If the numbers change, the rows in the database no longer match the file.

In [29]:
award_director = award_director.sort_values(["year_ceremony", "category", "film_title"]).reset_index(drop=True)

print(award_director[["year_ceremony", "category", "name_nominee", "film_title", "film_title_clean"]].head())

   year_ceremony                      category     name_nominee                      film_title                film_title_clean
0           1928    DIRECTING (Comedy Picture)  Lewis Milestone             Two Arabian Knights             two arabian knights
1           1928  DIRECTING (Dramatic Picture)    Frank Borzage                      7th Heaven                      7th heaven
2           1929                     DIRECTING      Frank Lloyd                 The Divine Lady                 the divine lady
3           1930                     DIRECTING  Lewis Milestone  All Quiet on the Western Front  all quiet on the western front
4           1931                     DIRECTING    Norman Taurog                          Skippy                          skippy


## STEP 8: give every award a director_award_id
director_award_id is the primary key of AWARD_Director. The database needs it to tell every row apart, so it must be unique and can never be empty.

In [30]:
award_director["director_award_id"] = award_director.index + 1

print("First director_award_id:", award_director["director_award_id"].min())
print("Last director_award_id:", award_director["director_award_id"].max())
print("All unique:", award_director["director_award_id"].is_unique)

First director_award_id: 1
Last director_award_id: 97
All unique: True


## STEP 9: add empty director_id and movie_id
Both are foreign keys in our ERD, to DIRECTOR and to MOVIE. The columns must exist to fit the table, but the numbers are not in the Oscar data. director_id is filled in the Directorclean notebook, movie_id later with the match on title and year.

In [31]:
award_director["director_id"] = None
award_director["movie_id"] = None

print(award_director.head())

   year_film  year_ceremony  ceremony                      category     name_nominee                      film_title  winner                film_title_clean                            match_key  \
0       1927           1928         1    DIRECTING (Comedy Picture)  Lewis Milestone             Two Arabian Knights    True             two arabian knights             two arabian knights_1927   
1       1927           1928         1  DIRECTING (Dramatic Picture)    Frank Borzage                      7th Heaven    True                      7th heaven                      7th heaven_1927   
2       1928           1929         2                     DIRECTING      Frank Lloyd                 The Divine Lady    True                 the divine lady                 the divine lady_1928   
3       1929           1930         3                     DIRECTING  Lewis Milestone  All Quiet on the Western Front    True  all quiet on the western front  all quiet on the western front_1929   
4       1930   

## STEP 10: keep only the ERD columns
The CSV must have exactly the same columns, as the table in PostgreSQL. The primary key comes first, then the foreign keys, so the most important columns are easy to find. winner is removed because every row is a winner, and year_film because it is always year_ceremony minus 1. Storing the same information twice goes against normalization. film_title_clean and match_key are not in the ERD. They stay in the file for now, because they are needed to fill movie_id. They are removed when movie_id is filled, before the import in pgAdmin.

In [32]:
award_director = award_director[["director_award_id", "director_id", "movie_id", "year_ceremony","ceremony", "category", "name_nominee", "film_title", "film_title_clean", "match_key"]]

print(list(award_director.columns))

['director_award_id', 'director_id', 'movie_id', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'film_title_clean', 'match_key']


## STEP 11: save as CSV
This file is imported into AWARD_Director in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table and the import would fail.

In [33]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_director_clean.csv")

award_director.to_csv(new_file, index=False)

print("Saved:", new_file)
print("File exists:", os.path.exists(new_file))

Saved: C:\Users\caple\OneDrive - HvA\Master\Data management\Data\award_director_clean.csv
File exists: True
